# Projekt: Predykcja emisji hałasu profilu lotniczego (Airfoil Self-Noise)
## Zastosowanie metod inteligencji obliczeniowej jako modelu zastępczego (Surrogate Model) dla symulacji numerycznych (BEM/CFD)

### 1. Wprowadzenie i cel projektu
W mechanice płynów oraz wibroakustyce numeryczne wyznaczanie pola akustycznego (np. za pomocą Metody Elementów Brzegowych - BEM rozwiązującej równanie falowe Helmholtza lub metod CFD) jest operacją wysoce kosztowną obliczeniowo.

**Cel projektu:**
Wykorzystanie metod analizy danych oraz uczenia maszynowego w języku Python do stworzenia szybkiego modelu predykcyjnego (tzw. *Surrogate Model* / cyfrowy bliźniak). Model ma za zadanie przewidywać poziom ciśnienia akustycznego (SSPL) generowanego przez profil skrzydła na podstawie parametrów aerodynamicznych i geometrycznych.


### 2. Pobranie danych
Do analizy wykorzystano zbiór danych NASA z repozytorium UCI Machine Learning Repository (`id=291`). Dane zostały pozyskane z oficjalnego API biblioteki `ucimlrepo`.

In [7]:
!pip install ucimlrepo -q

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

airfoil_self_noise = fetch_ucirepo(id=291)

X_raw = airfoil_self_noise.data.features
y_raw = airfoil_self_noise.data.targets

df = pd.concat([X_raw, y_raw], axis=1)

df.columns = [
    'Frequency',
    'Angle_of_attack',
    'Chord_length',
    'Free_stream_velocity',
    'Suction_side_displacement_thickness',
    'SSPL'
]

df.head()


,Frequency,Angle_of_attack,Chord_length,Free_stream_velocity,Suction_side_displacement_thickness,SSPL
0,800,0.0,0.3048,71.3,0.002663,126.201
1,1000,0.0,0.3048,71.3,0.002663,125.201
2,1250,0.0,0.3048,71.3,0.002663,125.951
3,1600,0.0,0.3048,71.3,0.002663,127.591
4,2000,0.0,0.3048,71.3,0.002663,127.461


### 3. Weryfikacja jakości danych i czyszczenie
W tym kroku sprawdzamy spójność typów danych, występowanie brakujących wartości (NaN) oraz podstawowe wymiary zbioru.

In [8]:
print(f"Liczba obserwacji (wierszy): {df.shape[0]}")
print(f"Liczba cech (kolumn): {df.shape[1]}")
print("\n--- Informacje o typach danych i brakach ---")
df.info()

print("\n--- Liczba brakujących wartości w poszczególnych kolumnach ---")
print(df.isnull().sum())

Liczba obserwacji (wierszy): 1503
Liczba cech (kolumn): 6

--- Informacje o typach danych i brakach ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 6 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Frequency                            1503 non-null   int64  
 1   Angle_of_attack                      1503 non-null   float64
 2   Chord_length                         1503 non-null   float64
 3   Free_stream_velocity                 1503 non-null   float64
 4   Suction_side_displacement_thickness  1503 non-null   float64
 5   SSPL                                 1503 non-null   float64
dtypes: float64(5), int64(1)
memory usage: 70.6 KB

--- Liczba brakujących wartości w poszczególnych kolumnach ---
Frequency                              0
Angle_of_attack                        0
Chord_length                           0
Free_stream_velocity    

Zbiór danych składa się z 1503 pomiarów. Wszystkie cechy są typu zmiennoprzecinkowego (`float64`) lub całkowitego (`int64`). Nie stwierdzono obecności brakujących wartości (NaN), co oznacza brak konieczności imputacji danych.

### 4. Inżynieria cech (Feature Engineering) z wiedzą dziedzinową
Standardowy zbiór zawiera wyłącznie proste wielkości wymiarowe. W mechanice płynów i wibroakustyce kluczową rolę odgrywają bezwymiarowe liczby kryterialne oraz charakterystyka logarytmiczna:

1. **Liczba Strouhala ($St$):** Opisuje zjawiska wiro-akustyczne oraz periodyczne odrywanie się wirów:
   $$St = \frac{f \cdot \delta}{U_\infty}$$
   gdzie: $f$ - częstotliwość, $\delta$ - grubość warstwy ściennej, $U_\infty$ - prędkość napływu.
2. **Liczba Reynoldsa ($Re$):** Opisuje stosunek sił bezwładności do sił lepkości:
   $$Re = \frac{U_\infty \cdot c}{\nu}$$
   gdzie: $c$ - długość cięciwy, $\nu \approx 1.5 \times 10^{-5} \, \text{m}^2/\text{s}$ (lepkość kinematyczna powietrza w warunkach normalnych).
3. **Logarytm częstotliwości ($\log_{10}(f)$):** Skala percepcji dźwięku oraz analizy widmowej ma charakter logarytmiczny.

In [9]:
# lepkosc kinematyczna powietrza w warunkach normalnych (ok. 20C)
nu = 1.5e-5

df['Strouhal_number'] = (df['Frequency'] * df['Suction_side_displacement_thickness']) / df['Free_stream_velocity']
df['Reynolds_number'] = (df['Free_stream_velocity'] * df['Chord_length']) / nu

# percepcja dzwieku i pasma tercjowe maja charakter wykladniczy
df['Log_Frequency'] = np.log10(df['Frequency'])

df[['Frequency', 'Strouhal_number', 'Reynolds_number', 'Log_Frequency']].head()

,Frequency,Strouhal_number,Reynolds_number,Log_Frequency
0,800,0.029884,1448816.0,2.90309
1,1000,0.037354,1448816.0,3.00000
2,1250,0.046693,1448816.0,3.09691
3,1600,0.059767,1448816.0,3.20412
4,2000,0.074709,1448816.0,3.30103


### 5. Przygotowanie danych pod zadania klasyfikacji i analizy grupowej
W celu umożliwienia późniejszej analizy grupowej (agregacja) oraz klasyfikacji binarnej, wprowadzono dwie cechy:
* **`Angle_regime`**: Podział kąta natarcia na zakresy (reżimy aerodynamiczne):
  * `Low` ($0^\circ - 5^\circ$) – mały kąt, stan stabilnego opływu,
  * `Medium` ($5^\circ - 10^\circ$) – stan pośredni,
  * `High` ($> 10^\circ$) – duży kąt, reżim separacji strugi / przeciągnięcia.
* **`High_Noise_Alert`**: Binarna flaga (0 / 1) określająca, czy wygenerowany hałas przekracza próg 130 dB (wartość krytyczna z punktu widzenia norm hałasu). Posłuży ona w dalszej fazie projektu do budowy modeli klasyfikacyjnych.

In [10]:
# podzial na rezimy aerodynamiczne umozliwi operacje groupby w dalszej czesci projektu
df['Angle_regime'] = pd.cut(
    df['Angle_of_attack'],
    bins=[-np.inf, 5.0, 10.0, np.inf],
    labels=['Low', 'Medium', 'High']
)

# prog 130 db jako granica przekroczenia norm halasu pod model klasyfikacji binarnej
df['High_Noise_Alert'] = (df['SSPL'] >= 130.0).astype(int)

print("Rozkład klas hałasu krytycznego (High_Noise_Alert):")
print(df['High_Noise_Alert'].value_counts(normalize=True))

df.head()

Rozkład klas hałasu krytycznego (High_Noise_Alert):
High_Noise_Alert
0    0.749834
1    0.250166
Name: proportion, dtype: float64


,Frequency,Angle_of_attack,Chord_length,Free_stream_velocity,Suction_side_displacement_thickness,SSPL,Strouhal_number,Reynolds_number,Log_Frequency,Angle_regime,High_Noise_Alert
0,800,0.0,0.3048,71.3,0.002663,126.201,0.029884,1448816.0,2.90309,Low,0
1,1000,0.0,0.3048,71.3,0.002663,125.201,0.037354,1448816.0,3.00000,Low,0
2,1250,0.0,0.3048,71.3,0.002663,125.951,0.046693,1448816.0,3.09691,Low,0
3,1600,0.0,0.3048,71.3,0.002663,127.591,0.059767,1448816.0,3.20412,Low,0
4,2000,0.0,0.3048,71.3,0.002663,127.461,0.074709,1448816.0,3.30103,Low,0


### 6. Wstępne statystyki opisowe
Zestawienie podstawowych miar statystycznych dla całego rozszerzonego zbioru danych przygotowanego do właściwej analizy eksploracyjnej i modelowania.

In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Frequency,1503.0,2886.380572,3152.573137,200.000000,800.000000,1600.000000,4000.000000,2.000000e+04
Angle_of_attack,1503.0,6.782302,5.918128,0.000000,2.000000,5.400000,9.900000,2.220000e+01
Chord_length,1503.0,0.136548,0.093541,0.025400,0.050800,0.101600,0.228600,3.048000e-01
Free_stream_velocity,1503.0,50.860745,15.572784,31.700000,39.600000,39.600000,71.300000,7.130000e+01
Suction_side_displacement_thickness,1503.0,0.011140,0.013150,0.000401,0.002535,0.004957,0.015576,5.841130e-02
SSPL,1503.0,124.835943,6.898657,103.380000,120.191000,125.721000,129.995500,1.409870e+02
Strouhal_number,1503.0,0.481073,0.878784,0.008308,0.074366,0.185322,0.477693,9.292707e+00
Reynolds_number,1503.0,463363.823464,360678.832198,53678.666667,134112.000000,402336.000000,724408.000000,1.448816e+06
Log_Frequency,1503.0,3.224877,0.471137,2.301030,2.903090,3.204120,3.602060,4.301030e+00
High_Noise_Alert,1503.0,0.250166,0.433253,0.000000,0.000000,0.000000,0.500000,1.000000e+00
